# Milestone - 1

### Objective : 
getting hands-on with Exploratory Data Analysis (EDA), text processing, and establishing baseline similarity metrics specifically for Natural Language Processing (NLP) pipelines.

-Text Processing: Writing code for tokenization, text normalization, and removing stop words to clean and prepare your raw text data.

-Vectorization & Similarity: Implementing TF-IDF (Term Frequency-Inverse Document Frequency) to convert your text into numerical vectors, and then calculating Cosine Similarity to measure how closely related different pieces of text are.

-Evaluation: Understanding and calculating Mean Average Precision (MAP@3) to evaluate the performance of your similarity metrics.

## Read The data and analyze it

In [17]:
import pandas as pd 
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
train.columns, test.columns

(Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object'),
 Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object'))

In [18]:
# shape and value count of answer column
print("Shape of test and train dataset")
print(f'train dataset shape : {train.shape}, test datatset shape : {test.shape}')
print()
print("Value count of answer column of train dataset")
frequency_of_option = train['answer'].value_counts()
print(frequency_of_option)
print()
# Q1. # Sum of most frequently and least frequently occure option 
sum_option = frequency_of_option.iloc[0] + frequency_of_option.iloc[-1]
print("Sum of least and most frequency option")
print(sum_option)

Shape of test and train dataset
train dataset shape : (2000, 8), test datatset shape : (500, 7)

Value count of answer column of train dataset
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Sum of least and most frequency option
814


##### 1. most frequent option is B and least is E. 
##### 2. Distribution of options in target column is noe so much imbalaced. 

## Handling missing values

In [19]:
print("Missing data in tarin & Test")
print(f'train : \n{train.isnull().sum()} , test : \n{test.isnull().sum()}')

Missing data in tarin & Test
train : 
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64 , test : 
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


##### The dataset does not contain missing values in either the training or test set. Therefore, no imputation or missing-value handling is required before preprocessing and feature extraction.

## Q2. Text Preprocessing & Vocabulary Analysis
"After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?"


In [20]:
import string 

# Function for cleaning row 
def clean_text(text) :
    text = text.lower()

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )
    return text

In [21]:
cleaned_prompt = train["prompt"].apply(clean_text)

# vocabulary set 
vocabulary = set() 

for prompt in cleaned_prompt :
    vocabulary.update(prompt.split())

print("Total unique words in prompt column")
print("Vocabulary Size:", len(vocabulary))

Total unique words in prompt column
Vocabulary Size: 859


##### "After converting all prompts to lowercase and removing standard punctuation characters, the total vocabulary size across the entire prompt column in train.csv was found to be 859 unique words."

"This observation shows that while raw natural language text can contain noise (such as mixed casing, punctuation marks, and symbols), standardizing the text reduces the total unique word count. This streamlined vocabulary helps in reducing feature space dimensionality before moving on to tokenization and vectorization phases.""

In [22]:
# Check for all columns prompt , A, B , C, D, E 
all_text = []

for col in ["prompt", "A", "B", "C", "D", "E"]:
    cleaned = train[col].apply(clean_text) 

    all_text.extend(cleaned)

full_vocab = set() 

for text in all_text :
    full_vocab.update(text.split())

print("vocabulary Size across the prompt and options Columns is " ,len(full_vocab))

vocabulary Size across the prompt and options Columns is  3096


## Q3. Tokenization & Stop Word Removal
"Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?"

In [23]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# let's see the first row of prompt 
row = train.loc[0, "prompt"]

clean_row = clean_text(row)

tokens = clean_row.split()
filter_tokens = [ word for word in tokens if word not in ENGLISH_STOP_WORDS]

print("after removing commom words, unique word in prompt at zero index : ",len(filter_tokens))
# i get 13 unique words  

after removing commom words, unique word in prompt at zero index :  13


##### "Filtering out standard English stop words from Row ID 1 reduced the noise significantly, leaving 13 informative tokens."

## Q4 : TfidfVectorizer
"Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?"

##### Now analyse the prompt and all options columns A, B , C , D , E

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer 

combined_text = (train["prompt"] + " "+
                 train["A"] + " " +
                 train["B"] + " "+
                 train["C"] + " "+
                 train["D"] + " "+
                 train["E"] )

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(combined_text)
print(X.shape)
print("after removing common words, Total unique words across options and promt columns are :  ",len(vectorizer.get_feature_names_out()))    

(2000, 2762)
after removing common words, Total unique words across options and promt columns are :   2762


The combined prompt and answer-option text produced a TF-IDF vocabulary of 2762 unique features. This indicates that incorporating answer choices substantially increases the available textual information compared to using prompts alone.

## Q5 
"Using the TF-IDF vectorizer fitted in Question 4, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places)"

In [25]:
print(train.loc[0, "prompt"])
print()
print(train.loc[0, "A"])

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.


In [26]:
# cosine similarity 
from sklearn.metrics.pairwise import cosine_similarity
prompt1 = train.loc[0,"prompt"]
option_a = train.loc[0,"A"]
option_b = train.loc[0,"B"]
option_c = train.loc[0,"C"]
option_d = train.loc[0,"D"]
option_e = train.loc[0,"E"]

prompt_vec = vectorizer.transform([prompt1])
option_a_vec = vectorizer.transform([option_a])
option_b_vec = vectorizer.transform([option_b])
option_c_vec = vectorizer.transform([option_c])
option_d_vec = vectorizer.transform([option_d])
option_e_vec = vectorizer.transform([option_e])

similarity_with_a = cosine_similarity(prompt_vec, option_a_vec)[0][0]
similarity_with_b = cosine_similarity(prompt_vec, option_b_vec)[0][0]
similarity_with_c = cosine_similarity(prompt_vec, option_c_vec)[0][0]
similarity_with_d = cosine_similarity(prompt_vec, option_d_vec)[0][0]
similarity_with_e = cosine_similarity(prompt_vec, option_e_vec)[0][0]

results = {
    "A": similarity_with_a,
    "B": similarity_with_b,
    "C": similarity_with_c,
    "D": similarity_with_d,
    "E": similarity_with_e
}

predicted = max(results, key=results.get)

print("similarity with a " ,similarity_with_a)
print("similarity with b ",similarity_with_b)
print("similarity with c ",similarity_with_c)
print("similarity with d ",similarity_with_d)
print("similarity with e ",similarity_with_e) 
print(predicted)

print(train.loc[0,'answer'])

similarity with a  0.27202429519891635
similarity with b  0.3036286931760937
similarity with c  0.5876662698435949
similarity with d  0.5385329310060211
similarity with e  0.23662770351943294
C
B


Each prompt and its corresponding answer options were transformed into TF-IDF vectors using the vocabulary learned from the training corpus. Cosine similarity was computed between the prompt vector and each option vector. The option with the highest similarity score was selected as the predicted answer, demonstrating a simple retrieval-based baseline for multiple-choice question answering.

## Q6 : 
"Expand the logic from Question 5: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options. Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer."

In [27]:
# cosine similarity for all rows 
correct_prediction = 0 

for idx, row in train.iterrows():
    prompt = row["prompt"]

    option_a = row["A"]
    option_b = row["B"]
    option_c = row["C"]
    option_d = row["D"]
    option_e = row["E"]

    actual_answer = row["answer"]

    prompt_vec = vectorizer.transform([prompt])

    option_a_vec = vectorizer.transform([option_a])
    option_b_vec = vectorizer.transform([option_b])
    option_c_vec = vectorizer.transform([option_c])
    option_d_vec = vectorizer.transform([option_d])
    option_e_vec = vectorizer.transform([option_e])

    scores = {
    "A": cosine_similarity(prompt_vec, option_a_vec)[0][0],
    "B": cosine_similarity(prompt_vec, option_b_vec)[0][0],
    "C": cosine_similarity(prompt_vec, option_c_vec)[0][0],
    "D": cosine_similarity(prompt_vec, option_d_vec)[0][0],
    "E": cosine_similarity(prompt_vec, option_e_vec)[0][0]
     }
    predicted_answer = max(scores, key=scores.get)

    if predicted_answer == actual_answer:
        correct_prediction += 1

accuracy = (correct_prediction / len(train)) * 100

print("Accuracy:", accuracy)

Accuracy: 13.55


The TF-IDF and cosine similarity approach achieved an accuracy of approximately 13.5%. This relatively low performance indicates that lexical similarity alone is insufficient for solving multiple-choice question answering tasks, as many correct answers require semantic understanding rather than simple word overlap.

## Map@3 Function Define

In [28]:
def map_at_3(actual, prediction) :
    if actual in prediction :
        rank = prediction.index(actual) + 1 

        return 1 / rank 
    return 0
# Test
print(map_at_3("C", ["C", "A", "B"]))
print(map_at_3("B", ["D", "B", "E"]))
print(map_at_3("A", ["D", "B", "C"]))

1.0
0.5
0


## Q9 :
Majority Class Baseline: Find the most frequent correct answer in the training set and calculate its overall MAP@3 score.

TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv using TF-IDF cosine similarity to sort options top-3, and find its final average MAP@3 score across the training set.

### Majority Baseline

In [29]:
## dummy prediction
scores =  []
for _,row in train.iterrows() :
    actual = row['answer'] 

    predict = ["B", "C", "A"]        # B : first most frequent answer , A : second most frequent , C :  third most frequent 

    scores.append(map_at_3(actual, predict))

final_map3 = sum(scores) / len(scores)

print(final_map3)

0.42125


### TFIDF Vectorizer Pipiline

In [30]:
tfidf_top3_predictions = []

scores = []

for _, row in train.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"]
    }

    prompt_vec = vectorizer.transform([prompt])

    similarities = {}

    for label, text in options.items():

        option_vec = vectorizer.transform([text])

        similarities[label] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    top3 = ranked[:3]

    tfidf_top3_predictions.append(top3)

    scores.append(
        map_at_3(
            row["answer"],
            top3
        )
    )

tfidf_map3 = sum(scores) / len(scores)
print("TF-IDF MAP@3 =", final_map3)

TF-IDF MAP@3 = 0.42125


The Majority Class Baseline achieved a MAP@3 score of 0.421, while the TF-IDF and cosine similarity approach achieved 0.290. Although TF-IDF utilizes textual information, it relies only on lexical overlap and does not capture semantic relationships between words. In this dataset, predicting the most frequent answer classes produced a stronger baseline than simple lexical matching.

In [31]:
import pickle

with open("tfidf_top3_predictions.pkl", "wb") as f:
    pickle.dump(tfidf_top3_predictions, f)

print("Saved TF-IDF predictions")

Saved TF-IDF predictions


### Observation :
- Dataset contains no missing values.
- Prompt vocabulary size = 859.
- Combined prompt + options vocabulary size = 2762.
- Majority class baseline MAP@3 = 0.42125.
- TF-IDF ranking baseline MAP@3 = 0.29617.
- TF-IDF relies on lexical overlap and does not capture semantic reasoning.
- More advanced transformer-based models are expected to significantly outperform TF-IDF.